In [ ]:
!pip install -q langchain langchain-core langchain-community \
langchain-text-splitters langchain-groq chromadb \
sentence-transformers ragas datasets pypdf

In [ ]:
import os

from langchain_groq import ChatGroq
from langchain_core.documents import Document

# fixed import
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision
)

from datasets import Dataset

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2186/3022934079.py:13: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_2186/3022934079.py:13: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics impo

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "******************"

In [ ]:
!pip install nbstripout
!nbstripout Untitled90.ipynb

Could not strip 'Untitled90.ipynb': file not found


In [ ]:
documents = [
Document(page_content="""
Retrieval-Augmented Generation (RAG) combines retrieval and generation.
It fetches relevant information before generating responses.
"""),

Document(page_content="""
RAGAS is used to evaluate RAG systems using metrics such as
faithfulness, answer relevancy and context precision.
"""),

Document(page_content="""
Chroma stores vector embeddings and supports similarity search.
""")
]

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

print("Number of chunks:", len(docs))

Number of chunks: 3


In [ ]:
embeddings = HuggingFaceEmbeddings(
model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2186/4074377248.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = Chroma.from_documents(
docs,
embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k":2})

print("Vector database created successfully")

Vector database created successfully


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print("LLM loaded successfully")

LLM loaded successfully


In [ ]:
query = "What is RAGAS?"

# Retrieve relevant documents
retrieved_docs = retriever.invoke(query)

# Combine retrieved context
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Prompt
prompt = f"""
Answer only using the provided context.

Context:
{context}

Question:
{query}
"""

# Generate answer
response = llm.invoke(prompt)

answer = response.content

print("Question:", query)
print("\nRetrieved Context:\n", context)
print("\nGenerated Answer:\n", answer)

Question: What is RAGAS?

Retrieved Context:
 RAGAS is used to evaluate RAG systems using metrics such as
faithfulness, answer relevancy and context precision.
Retrieval-Augmented Generation (RAG) combines retrieval and generation.
It fetches relevant information before generating responses.

Generated Answer:
 RAGAS is used to evaluate RAG systems.


In [ ]:
data = {
 "question":[query],
 "answer":[answer],
 "contexts":[[doc.page_content for doc in retrieved_docs]],
 "ground_truth":[
   "RAGAS is a framework used to evaluate Retrieval-Augmented Generation systems."
 ]
}

dataset = Dataset.from_dict(data)

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

/tmp/ipykernel_2186/2443030709.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_2186/2443030709.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [ ]:
result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print(result)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.7823, 'context_precision': 1.0000}
